# 04-04 高级 RAG 技术

**标准 RAG 的局限**：
- 用户 query 和文档在不同语义空间 → 检索不精准
- 单次检索无法回答多跳问题
- 检索到的文档不一定有用

**本节目标**：HyDE、Query Expansion、Self-RAG、Multi-hop RAG、Graph-RAG 概念

---

In [ ]:
import numpy as np
import sys, json
sys.path.insert(0, "..")
from utils.llm_client import call_llm
from utils.data_generator import generate_ad_knowledge_base

docs = generate_ad_knowledge_base()

def get_embeddings(texts):
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        return model.encode(texts, normalize_embeddings=True)
    except ImportError:
        embs = []
        for t in texts:
            rng = np.random.default_rng(hash(t) % (2**32))
            v = rng.standard_normal(128).astype(np.float32)
            embs.append(v / np.linalg.norm(v))
        return np.array(embs)

doc_embs = get_embeddings(docs)

def simple_retrieve(query: str, k: int = 3) -> list[tuple]:
    q_emb = get_embeddings([query])[0]
    sims = doc_embs @ q_emb
    top_k = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i]), docs[i]) for i in top_k]

print(f"知识库: {len(docs)} 个文档")

## 1. HyDE（Hypothetical Document Embeddings）

**核心思想**：用户 query 通常很短（"CTR怎么算"），与长文档的 embedding 空间不匹配。
HyDE 先让 LLM 生成一个「假想文档」，再用假想文档的 embedding 去检索。

```
标准 RAG:  query → embed(query) → 检索
HyDE:      query → LLM生成假想答案 → embed(假想答案) → 检索
```

In [ ]:
def hyde_retrieve(query: str, k: int = 3) -> list[tuple]:
    """
    HyDE 检索：先生成假想文档，再用其 embedding 检索
    """
    # Step 1: LLM 生成假想答案
    prompt = f"""请回答以下问题（即使不确定也要给出详细回答，约100字）：
问题：{query}"""
    
    try:
        hypothetical_doc = call_llm(prompt, max_tokens=200)
    except Exception:
        hypothetical_doc = f"关于{query}，通常涉及广告投放的核心指标计算，包括展示数、点击数和转化率的关系。"
    
    print(f"  假想文档: {hypothetical_doc[:80]}...")
    
    # Step 2: 用假想文档的 embedding 检索
    hyde_emb = get_embeddings([hypothetical_doc])[0]
    sims = doc_embs @ hyde_emb
    top_k = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i]), docs[i]) for i in top_k]

query = "CTR怎么计算"
print(f"查询: {query!r}\n")

print("标准检索:")
for idx, score, doc in simple_retrieve(query, k=2):
    print(f"  [{score:.3f}] {doc[:60]}...")

print("\nHyDE 检索:")
for idx, score, doc in hyde_retrieve(query, k=2):
    print(f"  [{score:.3f}] {doc[:60]}...")

## 2. Query Expansion（查询扩展）

In [ ]:
def expand_query(query: str, n: int = 3) -> list[str]:
    """
    用 LLM 生成多个等价查询，扩大检索覆盖面
    """
    prompt = f"""为以下问题生成{n}个不同的表述方式（换个角度问同样的问题），
每行一个，不要编号：

原始问题：{query}"""
    
    try:
        response = call_llm(prompt, max_tokens=200)
        expanded = [q.strip() for q in response.strip().split('\n') if q.strip()]
        return expanded[:n]
    except Exception:
        # Mock 扩展
        return [
            f"{query}的定义和公式",
            f"如何理解{query}",
            f"{query}的计算方法是什么",
        ][:n]


def multi_query_retrieve(query: str, k: int = 3) -> list[tuple]:
    """多查询检索：扩展 query → 每个 query 检索 → 合并去重"""
    expanded = expand_query(query)
    all_queries = [query] + expanded
    
    print(f"  扩展查询:")
    for q in all_queries:
        print(f"    - {q}")
    
    # 每个 query 检索 top-k，用 RRF 合并
    doc_scores = {}
    for q in all_queries:
        results = simple_retrieve(q, k=k)
        for rank, (idx, score, doc) in enumerate(results):
            # RRF: 1 / (rank + 60)
            rrf_score = 1.0 / (rank + 60)
            doc_scores[idx] = doc_scores.get(idx, 0) + rrf_score
    
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    return [(idx, score, docs[idx]) for idx, score in sorted_docs[:k]]

query = "如何提升广告效果"
print(f"查询: {query!r}\n")
results = multi_query_retrieve(query, k=3)
print(f"\n合并结果 Top-3:")
for idx, score, doc in results:
    print(f"  [RRF={score:.4f}] {doc[:60]}...")

## 3. Self-RAG（自适应检索）

Self-RAG 让模型在生成过程中**自己判断**是否需要检索、检索结果是否相关。

```
4 个反思 Token:
  [Retrieve] → 是否需要检索？（有些问题不需要 RAG）
  [ISREL]    → 检索到的文档是否和问题相关？
  [ISSUP]    → 生成的答案是否有文档支持？
  [ISUSE]    → 最终答案对用户是否有用？
```

In [ ]:
def self_rag_pipeline(query: str) -> dict:
    """
    Self-RAG 简化实现：
    1. 判断是否需要检索
    2. 检索并判断相关性
    3. 生成答案并验证忠实度
    """
    result = {"query": query, "steps": []}
    
    # Step 1: [Retrieve] 判断是否需要检索
    try:
        need_retrieve = call_llm(
            f"问题：'{query}'\n回答这个问题是否需要查阅外部资料？只回答'是'或'否'",
            max_tokens=10
        ).strip()
    except Exception:
        need_retrieve = "是"  # 默认需要检索
    
    result["steps"].append({"action": "[Retrieve]", "decision": need_retrieve})
    print(f"  [Retrieve] 需要检索? → {need_retrieve}")
    
    if "否" in need_retrieve:
        # 直接用 LLM 回答
        try:
            answer = call_llm(query, max_tokens=200)
        except Exception:
            answer = "这是一个常识性问题，无需检索即可回答。"
        result["answer"] = answer
        return result
    
    # Step 2: 检索 + [ISREL] 判断相关性
    retrieved = simple_retrieve(query, k=3)
    relevant_docs = []
    
    for idx, score, doc in retrieved:
        is_relevant = score > 0.3  # 简化：用相似度阈值判断
        tag = "RELEVANT" if is_relevant else "NOT_RELEVANT"
        result["steps"].append({"action": "[ISREL]", "doc_idx": idx, "decision": tag, "score": score})
        print(f"  [ISREL] doc_{idx} (sim={score:.3f}) → {tag}")
        if is_relevant:
            relevant_docs.append(doc)
    
    # Step 3: 生成答案 + [ISSUP] 验证
    if relevant_docs:
        context = "\n".join(relevant_docs)
        try:
            answer = call_llm(
                f"参考资料：\n{context}\n\n基于以上资料回答：{query}",
                max_tokens=200
            )
        except Exception:
            answer = f"基于检索到的{len(relevant_docs)}个相关文档，可以得出以下结论..."
        is_supported = True
    else:
        answer = "抱歉，检索到的资料与问题不够相关，建议联系平台客服获取准确信息。"
        is_supported = False
    
    result["steps"].append({"action": "[ISSUP]", "supported": is_supported})
    result["answer"] = answer
    print(f"  [ISSUP] 答案有文档支持? → {is_supported}")
    
    return result

print("=== Self-RAG Pipeline ===")
result = self_rag_pipeline("B站广告的出价方式有哪些")
print(f"\n最终答案: {result['answer'][:100]}...")

## 4. Multi-hop RAG（多跳检索）

In [ ]:
def multi_hop_rag(query: str, max_hops: int = 2) -> dict:
    """
    多跳 RAG：复杂问题分解为多个子查询，逐步检索
    适用于："对比X和Y的区别" "先查A再根据A查B" 等需要推理链的问题
    """
    all_context = []
    current_query = query
    
    for hop in range(max_hops):
        print(f"  Hop {hop+1}: 查询 '{current_query}'")
        
        # 检索当前子查询
        results = simple_retrieve(current_query, k=2)
        hop_docs = [doc for _, _, doc in results]
        all_context.extend(hop_docs)
        
        # 判断是否需要更多检索
        try:
            follow_up = call_llm(
                f"""原始问题：{query}
已检索到的信息：{' '.join(hop_docs)[:300]}

要完整回答原始问题，还需要查询什么信息？如果信息已充分，回复"DONE"。
否则给出下一个需要查询的子问题（一句话）。""",
                max_tokens=100
            )
        except Exception:
            follow_up = "DONE"
        
        if "DONE" in follow_up:
            print(f"  → 信息充分，停止检索")
            break
        else:
            current_query = follow_up.strip()
            print(f"  → 需要更多信息: '{current_query}'")
    
    # 用所有收集到的 context 生成最终答案
    full_context = "\n".join(set(all_context))  # 去重
    try:
        answer = call_llm(
            f"参考资料：\n{full_context[:1000]}\n\n综合以上信息，回答：{query}",
            max_tokens=300
        )
    except Exception:
        answer = f"综合{len(all_context)}条检索结果，对问题进行多角度分析..."
    
    return {"query": query, "hops": min(hop+1, max_hops), "context_docs": len(all_context), "answer": answer}

print("=== Multi-hop RAG ===")
result = multi_hop_rag("B站广告CPM和CPC的区别以及各自的适用场景")
print(f"\n跳数: {result['hops']}, 检索文档: {result['context_docs']}")
print(f"答案: {result['answer'][:150]}...")

## 5. Graph-RAG 概念

**Graph-RAG**（微软提出）：将文档转化为知识图谱，利用图结构做检索。

```
标准 RAG:  文档 → chunks → embed → 向量检索
Graph-RAG: 文档 → 实体/关系提取 → 知识图谱 → 社区检测 → 层级摘要 → 检索
```

### Graph-RAG 流程

1. **实体提取**: LLM 从文档中提取实体（广告主、广告位、指标）和关系
2. **构建图**: 实体为节点，关系为边
3. **社区检测**: Leiden 算法发现聚类（如"计费相关""素材相关"群组）
4. **层级摘要**: 每个社区由 LLM 生成摘要
5. **检索**: query → 匹配相关社区 → 利用社区摘要回答

### 适用场景

| 场景 | 标准 RAG | Graph-RAG |
|------|---------|----------|
| 点查（"X是什么"） | ✅ 足够 | 过度设计 |
| 全局分析（"总结所有广告类型"） | ❌ 难以覆盖 | ✅ 社区摘要擅长 |
| 多跳推理（"A和B什么关系"） | ❌ 需要多次检索 | ✅ 图上直接遍历 |
| 成本 | 低 | 高（需要预处理构图） |

In [ ]:
# Graph-RAG 的实体提取示例（概念演示）
sample_doc = "B站广告支持CPM、CPC、OCPM三种出价方式。CPM最低出价5元，适合品牌曝光。CPC按点击计费，适合效果广告。OCPM智能优化，适合转化目标。"

print("=== Graph-RAG 实体提取示例 ===")
print(f"输入文档: {sample_doc}")

# 模拟提取的实体和关系
entities = [
    {"name": "CPM",  "type": "计费方式", "description": "千次展示计费"},
    {"name": "CPC",  "type": "计费方式", "description": "点击计费"},
    {"name": "OCPM", "type": "计费方式", "description": "智能千次展示计费"},
    {"name": "品牌曝光", "type": "目标"},
    {"name": "效果广告", "type": "目标"},
    {"name": "转化",   "type": "目标"},
]

relations = [
    ("CPM",  "适合", "品牌曝光"),
    ("CPC",  "适合", "效果广告"),
    ("OCPM", "适合", "转化"),
    ("CPM",  "最低出价", "5元"),
]

print(f"\n提取的实体 ({len(entities)}):")
for e in entities:
    print(f"  [{e['type']}] {e['name']}")

print(f"\n提取的关系 ({len(relations)}):")
for s, r, o in relations:
    print(f"  {s} --[{r}]--> {o}")

print("\n这些实体和关系构成知识图谱，Graph-RAG 就在此图上做检索。")

## 面试速记

| 问题 | 要点 |
|------|------|
| HyDE 原理 | query→LLM生成假想答案→embed假想答案→检索。让query进入"文档空间" |
| Self-RAG 4个token | Retrieve(是否检索), ISREL(文档相关?), ISSUP(有支撑?), ISUSE(有用?) |
| 什么时候用 Graph-RAG | 全局总结、多跳推理、实体关系查询；点查用标准RAG更高效 |
| RAG 评估指标 | Faithfulness(忠实度)、Answer Relevancy(相关性)、Context Precision/Recall |
| Query Expansion 的好处 | 多角度查询覆盖更多相关文档，用 RRF 融合排名 |

**下一节**: `../05-langchain-langgraph/01_langchain_basics.ipynb`